# Convolutional Neural Networks


## Setup

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image

import random
import numpy as np

In [11]:
# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Custom AlexNet

In [2]:
class CNNNet(nn.Module):

    def __init__(self, num_classes=2):
        super(CNNNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2), # Note, better to use AdaptiveMaxPool2d
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

cnnnet = CNNNet()

## Utils

Train function:

In [3]:
def train(model, optimizer, loss_fn, train_loader, val_loader, epochs=20, device="cpu"):
    for epoch in range(1, epochs+1):
        training_loss = 0.0
        valid_loss = 0.0

        # TRAIN
        model.train()
        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            output = model(inputs)
            loss = loss_fn(output, targets)
            loss.backward()
            optimizer.step()

            training_loss += loss.item() * inputs.size(0)

        training_loss /= len(train_loader.dataset)

        # VALIDATION
        model.eval()
        num_correct = 0
        num_examples = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device)

                output = model(inputs)
                loss = loss_fn(output, targets)

                valid_loss += loss.item() * inputs.size(0)

                preds = torch.argmax(output, dim=1)
                num_correct += (preds == targets).sum().item()
                num_examples += targets.size(0)

        valid_loss /= len(val_loader.dataset)

        print(f"Epoch: {epoch}, Training Loss: {training_loss:.2f}, "
              f"Validation Loss: {valid_loss:.2f}, Accuracy: {num_correct / num_examples:.2f}")

## Load Data

In [6]:
import torchvision
from torch.utils.data import Dataset

class BinaryCatFrogCIFAR10(Dataset):
    """
    0 -> cat
    1 -> frog
    """
    def __init__(self, root="./data", train=True, transform=None, download=True):
        self.base = torchvision.datasets.CIFAR10(
            root=root,
            train=train,
            download=download
        )
        self.transform = transform

        classes = self.base.classes

        self.cat_idx = classes.index("cat")
        self.frog_idx = classes.index("frog")

        self.samples = []
        for i, target in enumerate(self.base.targets):
            if target == self.cat_idx:
                self.samples.append((i, 0))
            elif target == self.frog_idx:
                self.samples.append((i, 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        base_idx, label = self.samples[idx]
        image, _ = self.base[base_idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [7]:
img_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_train_dataset = BinaryCatFrogCIFAR10(
    root="./data",
    train=True,
    transform=img_transforms,
    download=True
)

test_data = BinaryCatFrogCIFAR10(
    root="./data",
    train=False,
    transform=img_transforms,
    download=True
)

# Split train into train/val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_data, val_data = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)


100%|██████████| 170M/170M [00:10<00:00, 15.6MB/s]


## Training

In [13]:
cnnnet.to(device)
optimizer = optim.Adam(cnnnet.parameters(), lr=0.001)

batch_size=64
train_data_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size)
val_data_loader  = torch.utils.data.DataLoader(val_data, batch_size=batch_size)
test_data_loader  = torch.utils.data.DataLoader(test_data, batch_size=batch_size)

In [14]:
train(cnnnet, optimizer,torch.nn.CrossEntropyLoss(), train_data_loader,val_data_loader, epochs=1, device=device)

Epoch: 1, Training Loss: 0.62, Validation Loss: 0.46, Accuracy: 0.79


## Downloading a pretrained network

There are two ways of downloading pre-trained image models with PyTorch: `torchvision.models` library, or PyTorch Hub - preferred

In [ ]:
import torchvision.models as models
alexnet = models.alexnet(num_classes=1000, pretrained=True)

In [ ]:
resnet50 = torch.hub.load('pytorch/vision', 'resnet50')

In [ ]:
print(alexnet)

In [ ]:
print(resnet50)